# Supplementary Note 3 - Additional results

Figures S23–S33 and Supplementary Table S1: the residual load before event detection, the threshold, and the parts of the extremes analysis the main figures do not carry.

In [ ]:
import pandas as pd

from cordex2energy_analysis.config import (DURATION_BINS, RUN_TAG_HIGH_2050, SEASON_MAP, SEASON_MONTHS,
                                           SEASON_ORDER, THRESHOLD_PERCENTILE, EVENT_DETECTION,
                                           ZONE_LABELS, ZONE_ORDER, setup)
from cordex2energy_analysis.extremes_analysis import event_anomaly_vs_hourly_climatology
from cordex2energy_analysis.extremes_thresholds import calibrate_seasonal_active_thresholds
from cordex2energy_analysis.fields import load_country_output, load_demand_by_month
from cordex2energy_analysis.figures_results import (figure_s23, figure_s24, figure_s25, figure_s26,
                                                    figure_s27, figure_s28, figure_s29, figure_s30,
                                                    figure_s31, figure_s32, figure_s33)
from cordex2energy_analysis.plotting_publication import (europe_change_summary, europe_change_table,
                                                         markdown_table, publication_rc, save)
from cordex2energy_analysis.power_system import REALTIME_BALANCING_ZONES
from cordex2energy_analysis.runs import (cerra_system_metrics, event_tables, exceedance_by_hour,
                                         load_run, reference_period_thresholds,
                                         reservoir_hydro_by_season, seasonal_event_counts)
from cordex2energy_analysis.tables import (EUROPE_COLUMNS, duration_spectrum, europe_change_rows,
                                           event_anomaly, paired_percent_change, zones_with_events)

context = setup("si_note3_results")

In [ ]:
run = load_run(context)                          # the paper's run, scored on its CERRA p95
tables = event_tables(run, context)
hour_exceedance = exceedance_by_hour(run, context)
high_run = load_run(context, RUN_TAG_HIGH_2050)          # HIGH/2050 with its own CERRA p95
high_on_paper_threshold = load_run(context, RUN_TAG_HIGH_2050, threshold=run.threshold)
high_events_by_season = seasonal_event_counts(high_on_paper_threshold, context)

## S23 - Residual load by season
**In:** per-country residual load of the paper's run. **Shows:** each season's level at +1.5 °C and its change at +2.0 and +3.0 °C.

In [ ]:
with publication_rc():
    save(figure_s23(load_country_output(context, files=("residual_load.nc",)), context),
         "S23_residual_load_seasonal", context.figure_dir)

## S24 and Supplementary Table S1 - The Europe-wide change by balancing zone
**In:** the Figure 4 quantities summed over each zone's countries. **Shows:** each chain's change from +1.5 to +3.0 °C per zone, annually, in winter and in summer (Table S1 lists the values).

In [ ]:
europe_rows = europe_change_rows(load_country_output(context), load_demand_by_month(context),
                                 context.period_windows)
europe_labels = [label for label, _ in europe_rows]
zone_change = {zone: europe_change_table(europe_rows, EUROPE_COLUMNS, context.reference,
                                         context.warmest, members=REALTIME_BALANCING_ZONES[zone])
               for zone in ZONE_ORDER}
with publication_rc():
    save(figure_s24(zone_change, europe_labels, EUROPE_COLUMNS), "S24_europe_change_by_zone",
         context.figure_dir)

table_s1 = {column: pd.concat({ZONE_LABELS[zone]: europe_change_summary(
                                   zone_change[zone], europe_labels, EUROPE_COLUMNS)[column]
                               for zone in ZONE_ORDER}, names=["zone", "quantity"])
            for column in EUROPE_COLUMNS}
pd.concat(table_s1, names=["time scale"]).to_csv(
    context.figure_dir / "table_s1_europe_change_by_zone.csv", float_format="%.2f")
for column, part in table_s1.items():
    print(f"Supplementary Table S1 - {column}\n{markdown_table(part)}\n")

## S25 - CORDEX against CERRA
**In:** each chain's residual load in its +1.5 °C window and the CERRA pass. **Shows:** their distributions per zone (a) and the p95 threshold each would imply (b).

In [ ]:
reference_metrics, chain_thresholds = reference_period_thresholds(run, context)
with publication_rc():
    figure, threshold_bias = figure_s25(reference_metrics, chain_thresholds,
                                        cerra_system_metrics(context), run.threshold, run.zones)
    save(figure, "S25_cordex_vs_cerra", context.figure_dir)
threshold_bias

## S26 - Season-specific thresholds
**In:** the CERRA positive residual load. **Shows:** the p95 of each zone and season, which the paper does not use.

In [ ]:
seasonal_thresholds, _ = calibrate_seasonal_active_thresholds(
    run.cerra_deficit, run.zones, SEASON_ORDER, SEASON_MONTHS, "PERCENTILE",
    percentile=THRESHOLD_PERCENTILE)
with publication_rc():
    save(figure_s26(seasonal_thresholds, run.zones), "S26_threshold_calibration", context.figure_dir)

## S27 - The paper's threshold inside HIGH/2050
**In:** the CERRA pass of the HIGH/2050 build-out. **Shows:** its deficit survival curve per zone, with the paper's p95 and HIGH's own.

In [ ]:
with publication_rc():
    save(figure_s27(high_run, run.threshold, run.zones), "S27_threshold_placement", context.figure_dir)

## S28 - The deficit tail, every zone and season
**In:** zonal residual load of the paper's run. **Shows:** exceedance curves per zone and season, with the CERRA p95.

In [ ]:
with publication_rc():
    save(figure_s28(run, context, tables["events_by_season"]), "S28_tail_all_seasons",
         context.figure_dir)

## S29 - The duration spectrum by zone
**In:** every detected event and its severity. **Shows:** events (a) and severity (b) per year by duration, per zone.

In [ ]:
zone_spectrum = {value: duration_spectrum(tables["storage_events"], value, by=("model", "period", "zone"))
                 for value in ("counts", "severity")}
with publication_rc():
    save(figure_s29(zone_spectrum, context.style), "S29_duration_spectrum_by_zone", context.figure_dir)

## S30 - The diurnal cycle and its drivers, zone by zone
**In:** exceedance by hour of day and the driver anomalies over the deficit window. **Shows:** Figure 6's second row for every zone.

In [ ]:
with publication_rc():
    save(figure_s30(hour_exceedance, tables["evening_peak_components"], context),
         "S30_diurnal_and_drivers_by_zone", context.figure_dir)

## S31 - The multi-day regime
**In:** multi-day events and their seasonal-mean driver anomalies. **Shows:** events per zone, their seasonal change and the driver decomposition, as Figure 6 for sub-daily events.

In [ ]:
with publication_rc():
    save(figure_s31(tables, context), "S31_multi_day_regime", context.figure_dir)

## S32 - Why Iberian storage balances less under warming
**In:** Iberia's storage shares and its reservoir inflow and dispatch. **Shows:** the balanced share per warming level (a) and the seasonal change in inflow and dispatch (b).

In [ ]:
hydro_change = paired_percent_change(reservoir_hydro_by_season(run, context, "Iberian_Peninsula"),
                                     context.reference, context.warmest)
with publication_rc():
    save(figure_s32(tables["storage_attenuation"], hydro_change, context), "S32_iberia_attenuation",
         context.figure_dir)

## S33 - What survives the HIGH/2050 build-out
**In:** HIGH/2050 scored on the paper's threshold, and the event anomalies against the hour-of-day climatology. **Shows:** surviving events per zone and season (a, b) and their winter drivers (c, d).

In [ ]:
anomaly_table = event_anomaly_vs_hourly_climatology(
    high_run.residual_load, high_run.components, high_on_paper_threshold.threshold_by_model,
    context.period_windows, run.zones, SEASON_MAP, method=EVENT_DETECTION, max_duration_h=24.0)
anomaly = lambda block, zone, season: event_anomaly(anomaly_table, block, zone, season,
                                                    context.warmest, context.style.drivers)
survivors = zones_with_events(high_events_by_season, run.zones, DURATION_BINS,
                              context.period_names)
with publication_rc():
    save(figure_s33(high_events_by_season, anomaly, survivors, context),
         "S33_high_buildout_survivors", context.figure_dir)